# 10 Export: Charts 1–7: the promise and the reckoning

**Owner:** Sahil

Writes one upload-ready CSV per chart to `outputs/charts/`, its spec entry to `outputs/charts/chart_specs.md`, and the numbers its title uses to `outputs/findings.json` (`export_chart_NN`). Chart tables are built by `src/exports.py` from `data/clean/analysis.parquet` and the notebook 05 findings. Titles are assembled from those numbers, never typed by hand.

Run notebook 05 first. Featured five: `findings.json → featured_five`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src import config, exports as ex, findings as fd

pd.set_option("display.max_columns", None); pd.set_option("display.width", 220)
NB = "10_export_act1_2"
P = config.PALETTE
ctx = ex.load_ctx()
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append({"Check": name, "Result": "PASS" if passed else "FAIL", "Detail": detail})
    print(("PASS" if passed else "FAIL"), "-", name, ("| " + detail if detail else ""))

def publish(n, short, df, facts, spec):
    check(f"Chart {n:02d}: under {config.CHART_MAX_ROWS} rows", len(df) <= config.CHART_MAX_ROWS, f"{len(df)} rows")
    ex.save_csv(n, short, df)
    fd.update(f"export_chart_{n:02d}", facts, NB)
    ex.write_spec(n, spec)
    print(f"wrote outputs/charts/{n:02d}_{short}.csv ({len(df)} rows) and its spec")
    display(df.head(8))
print("Featured five:", list(ctx.featured_names.values()))

## Chart 01: wonderkid_table

In [ ]:
df, f = ex.chart_01(ctx)
spec = ex.spec_entry(1, "wonderkid_table", "Datawrapper table (search on)",
    f"Only {f['reached']} of FIFA 15's {f['n']} wonderkids reached their potential in the game, but {f['made_it']} became Big-5 regulars",
    "Under-21s rated potential 85+ in FIFA 15 (Sept 2014). Peak later rating = best overall in FIFA 16 to FC 24. Made it = 3,000+ Big-5 league minutes after Sept 2014",
    "columns in file order; Player bold; sort by Potential (desc); search on; 15 rows per page",
    f"Status text colours: Reached {P['reality']}, Fell short {P['fell_short']}, Vanished {P['context']}; Featured rows: background tint of {P['promise']} at 15% opacity; Gap as a bar column (red {P['fell_short']} for positive)",
    "(tables have no tooltip)",
    "Use Datawrapper's 'Highlight rows' for Featured = Yes. Make 'Made it' a coloured text column (Yes = reality green). Hide the Featured column after highlighting. Peak market value in € millions.")
publish(1, "wonderkid_table", df, f, spec)

## Chart 02: promise_inflation

In [ ]:
df, f = ex.chart_02(ctx)
spec = ex.spec_entry(2, "promise_inflation", "Datawrapper column chart",
    f"The game handed out {f['multiple']}× more wonderkid ratings in five years: from {f['wk_per_1000_first']} to {f['wk_per_1000_max']} per 1,000 under-21s by {f['wk_per_1000_max_edition']}",
    f"Under-21s rated potential 85+ per 1,000 under-21s in each edition. Over the same years average under-21 potential rose {f['mean_potential_change_first_to_max']} points and average overall only {f['mean_overall_change_first_to_max']}",
    "x = Edition; y = Wonderkids per 1,000 under-21s; other columns tooltip only",
    f"bars {P['promise']}; highlight the peak edition bar with a darker outline; axis/text {P['text']}",
    "<b>{{ Edition }}</b><br>{{ Wonderkids per 1,000 under-21s }} per 1,000 under-21s<br>{{ Wonderkids (under-21, potential 85+) }} wonderkids of {{ Under-21s in game }} under-21s<br>Average headroom: {{ Average headroom (potential − overall) }} points",
    "Keep editions as text categories in order. Add an annotation on the peak bar. Optional second panel: line of 'Average headroom (potential − overall)'. Population counts, not a sample, so no error bars.")
publish(2, "promise_inflation", df, f, spec)

## Chart 03: rating_trajectories

In [ ]:
panel = pd.read_parquet(config.CLEAN / "fifa_panel.parquet", columns=["player_id", "edition", "overall", "edition_date"])
df, f = ex.chart_03(ctx, panel)
spec = ex.spec_entry(3, "rating_trajectories", "Flourish line chart",
    "Five FIFA 15 prospects, five different endings",
    "Overall rating in each edition, FIFA 15 (2014) to FC 24 (2023). Grey: median of FIFA 15's wonderkids",
    "x = Edition (category); one line per player column; 'FIFA 15 wonderkids (median)' as a context line",
    f"featured players {P['promise']} with end-of-line name labels (vary weight or dash so they stay distinguishable); median line {P['context']} dashed",
    "<b>{{ series }}</b> · {{ Edition }}<br>Overall rating: {{ value }}",
    "Blank cells mean the player isn't in that edition: turn off 'Connect gaps' so Živković's line visibly stops. y axis 60–95. Label each line at its last point.")
publish(3, "rating_trajectories", df, f, spec)

## Chart 04: promise_vs_reality

In [ ]:
df, f = ex.chart_04(ctx)
spec = ex.spec_entry(4, "promise_vs_reality", "Datawrapper scatter plot (moved from Flourish: needs reference lines)",
    f"Every 10 points of potential delivered only {f['points_per_10']} points of real rating",
    f"{f['n']:,} prospects with potential 75+ (FIFA 15–18 entrants). x = potential at entry, y = best later overall. {f['below_diagonal_pct']}% sit below the dashed line, where the promise was met",
    "x = Potential; y = Peak later rating; colour = Status; label = Player where Featured = Yes; tooltip = all columns",
    f"Reached {P['reality']}, Fell short {P['fell_short']} (dots at 40% opacity, size 4); Featured dots {P['promise']}, size 8, labelled",
    "<b>{{ Player }}</b> ({{ Entry edition }}, {{ Position group }})<br>Potential: {{ Potential }}<br>Best later rating: {{ Peak later rating }}<br>{{ Club }} · {{ Nation }}",
    f"Add two custom lines (Annotate > Custom lines): 'Promise met' y = x from (75,75) to (95,95), dashed {P['text']}; 'What potential actually delivered' from (75,{f['line_at_75']}) to (95,{f['line_at_95']}), solid {P['reality']} (calibration line: peak = {f['line_intercept']} + {f['line_slope']} × potential, fitted on all entrants with a later edition). Slope {f['slope']} (95% CI {f['slope_ci'][0]}–{f['slope_ci'][1]}); under alternative survivorship treatments it ranges {f['survivor_slope_range'][0]}–{f['survivor_slope_range'][1]}, never near 1. Axes 60–95 (y) and 75–95 (x). Many dots share integer coordinates: use low opacity or Datawrapper's jitter.")
publish(4, "promise_vs_reality", df, f, spec)

## Chart 05: game_vs_reality

In [ ]:
df, f = ex.chart_05(ctx)
spec = ex.spec_entry(5, "game_vs_reality", "Datawrapper range plot (arrow)",
    "The bigger the promise, the less often the number came true, and the more often the career did",
    f"Share of each potential band (FIFA 15–18 entrants aged 21 or under) that reached its potential in the game vs made it in real life (3,000+ Big-5 league minutes). 85+: {f['top_band_reached_pct']}% vs {f['top_band_made_it_pct']}%; under 70: {f['bottom_band_reached_pct']}% vs {f['bottom_band_made_it_pct']}%",
    "rows = Potential band (keep order <70 … 85+); range start = Reached potential in the game (%); range end = Made it in real life (%); arrows on",
    f"start dot {P['promise']} (the game's promise), end dot {P['reality']} (reality); arrow colour by direction (up = reality green, down = {P['fell_short']})",
    "<b>Potential {{ Potential band }}</b> ({{ Players }} players)<br>Reached potential in the game: {{ Reached potential in the game (%) }}%<br>Made it in real life: {{ Made it in real life (%) }}%<br>Fell short: {{ Fell short in the game (%) }}% · Vanished: {{ Vanished from the game (%) }}%",
    "x axis 0–100%. Label both ends with values. The crossing point (between 70–74 and 75–79) is the story: say so in an annotation.")
publish(5, "game_vs_reality", df, f, spec)

## Chart 06: busts_and_bloomers

In [ ]:
df, f = ex.chart_06(ctx)
b, bl = f["biggest_bust"], f["biggest_bloomer"]
spec = ex.spec_entry(6, "busts_and_bloomers", "Datawrapper range plot",
    f"{b['player']} fell {b['gap']} points short of his promise; {bl['player']} beat his by {abs(bl['gap'])}",
    "Busts: the 10 wonderkids (potential 85+) who fell furthest short. Bloomers: the 10 biggest overachievers among players who reached a peak of 80+",
    "rows = Player; range start = Potential at entry; range end = Peak later rating; group by Group (Bust / Bloomer)",
    f"Potential dot {P['promise']}; peak dot {P['fell_short']} for busts, {P['beat']} for bloomers",
    "<b>{{ Player }}</b> · {{ Entry edition }}, {{ Club at entry }}<br>Potential: {{ Potential at entry }} → best later: {{ Peak later rating }} ({{ Gap }})",
    "Sort busts by gap (largest first), then bloomers by gap. Use Datawrapper's row grouping (a 'Group' column split) or two stacked charts. x axis 50–95.")
publish(6, "busts_and_bloomers", df, f, spec)

## Chart 07: did_ea_learn

In [ ]:
df, f = ex.chart_07(ctx)
spec = ex.spec_entry(7, "did_ea_learn", "Flourish line chart",
    f"As the promises grew, fewer came true: {f['first']}% of FIFA 15 entrants reached theirs within four editions, {f['last']}% by FIFA 20",
    f"Share of each entry cohort reaching its potential within the next 4 editions. Dashed: only players present in all 4 editions ({f['first_surv']}% → {f['last_surv']}%), so drop-outs don't drive the trend",
    "x = Entry edition; lines = 'Reached potential within 4 editions (%)' and 'Same, players present in all 4 editions (%)'; tooltip adds Mean gap within 4 editions",
    f"main line {P['reality']}; survivors-only line {P['reality']} dashed; annotate the inflation peak (FIFA 20) in {P['promise']}",
    "<b>{{ Entry edition }}</b><br>Reached within 4 editions: {{ value }}%<br>Mean gap: {{ Mean gap within 4 editions }} points",
    "y axis 0–50%. Pair with chart 2 in the text: the promise inflated while the hit rate fell.")
publish(7, "did_ea_learn", df, f, spec)

## Summary

In [ ]:
checks = pd.DataFrame(CHECKS); display(checks)
assert (checks["Result"] == "PASS").all()